# TL-Bot - char_classifier Training (Kaggle only)

**Before the very first run:**
- Enable GPU (*Settings → Accelerator → GPU T4 x2*)
- Add a Kaggle Secret named `RCLONE_TOKEN` (see below)

**Every session: run all cells top to bottom.**
- Cell 1 — set scripts, epoch count, and dataset variant.
- Cell 2 — installs/configures rclone and syncs checkpoints + dataset in from Google Drive.
- Cell 3 — clones/pulls the repo, symlinks the synced dataset in, then starts or resumes training. Checkpoints auto-sync back to Drive every 10 minutes during training and once on finish.
- Cell 4 — final results once cell 3 completes: run summary, the metrics of the epoch saved as `best.pt`, and `curves.png`. Refuses to report on an unfinished run.

Checkpoints persist across sessions via the same Google Drive folder Colab uses.

---
**Dataset zips already on Drive:** `char-dataset.zip` (original, runs 1-4) and `char-dataset-ctx-small.zip` (run 5 comparison — string-rendered + target-glyph-cropped tiles, count-parity with the original) are both uploaded to `My Drive/Colab Notebooks/TL-Bot/`. Cell 1's `DATASET_NAME` picks which one this session trains against.

**One-time: zip and upload a new/different dataset variant**
```powershell
.venv\Scripts\python.exe Models\remote_train.py --zip-dataset --dataset-name <name>
# Upload the resulting <name>.zip to My Drive/Colab Notebooks/TL-Bot/
```

**One-time: Kaggle rclone setup**
```powershell
# 1. Configure rclone (if not already done)
rclone config   # -> New remote -> name: gdrive -> type: Google Drive -> follow OAuth flow

# 2. Copy the token JSON to your clipboard
(Get-Content "$env:APPDATA\rclone\rclone.conf" | Select-String "^token = ").ToString().Replace("token = ", "") | Set-Clipboard

# 3. Add a Kaggle Secret: Add-ons -> Secrets -> Add new secret
#    Name: RCLONE_TOKEN   Value: paste clipboard (the access-token JSON)
```

To force a checkpoint push after a crash:
`!rclone sync /kaggle/working/TL-Bot/checkpoints "gdrive:Colab Notebooks/TL-Bot/checkpoints/"`

This notebook is Kaggle-only. For Colab/Lightning AI/Local runs, use `colab_train.ipynb` / `lightning_train.ipynb` / `local_train.ipynb` instead.

---
## Cell 1 - Configure
Set scripts and epoch count for this run. Edit here only.

In [ ]:
SCRIPTS = ["latin"]

# Epochs for this run.
# RESUME defaults to True -- checking progress.json before each session (cell 3
# prints it automatically) is how to decide whether continuing is worthwhile or
# hyperparameters need a change, not a hardcoded per-script flag. best.pt only
# ever updates on a genuine score improvement (see train.py), so it can't
# regress from a bad resume or a bad fresh attempt either way.
EPOCHS = 60

SCHEDULER = "cosine"   # cosine (recommended) | cosine-warm | none
LR        = 3e-4       # head LR; backbone uses LR * 0.1
RESUME    = True        # resume last.pt; set to False only for a deliberate fresh restart

# Dataset variant to train against. Defaults to "char-dataset-ctx-small" --
# run 5, the string-rendered + target-glyph-cropped tiles (see
# render_chars_context.py) at count-parity with the original char-dataset
# (77,465 vs 77,799 images, tile_size=64) -- this is the comparison run
# against run 4 (0.4814 @ epoch 15, char-dataset, crop-scale 0.40-0.75).
# Set back to "char-dataset" to reproduce/continue runs 1-4 instead. Must
# already be uploaded to My Drive/Colab Notebooks/TL-Bot/<DATASET_NAME>.zip:
#   python Models/remote_train.py --zip-dataset --dataset-name <name>
# A non-default name automatically gets its own checkpoints/<script>_<suffix>
# dir (mirrors remote_train.py's _make_ckpt_dir), so a comparison run can
# never land in and overwrite the default dataset's checkpoint.
DATASET_NAME = "char-dataset-ctx-small"

# Kaggle: rclone syncs checkpoints and dataset to/from the same Drive folder as Colab.
KAGGLE_ROOT = "/kaggle/working/TL-Bot"

---
## Cell 2 - Setup
Installs rclone, configures the gdrive remote from the RCLONE_TOKEN secret, and syncs checkpoints + dataset in from Google Drive.

In [ ]:
import os
from kaggle_secrets import UserSecretsClient
from pathlib import Path
import subprocess
import zipfile as _zipfile

# Install rclone if not already present
_which = subprocess.run(["which", "rclone"], capture_output=True)
if _which.returncode != 0:
    print("Installing rclone ...")
    subprocess.run(["curl", "-fsSL", "https://rclone.org/install.sh",
                    "-o", "/tmp/rclone_install.sh"], check=True)
    subprocess.run(["sudo", "bash", "/tmp/rclone_install.sh"], check=True)
    print("rclone installed.")
else:
    print(f"rclone already installed: {_which.stdout.decode().strip()}")

# Configure gdrive via env vars -- no config file needed.
# RCLONE_TOKEN = the access-token JSON from the "token = " line of rclone.conf.
os.environ["RCLONE_CONFIG_GDRIVE_TYPE"]  = "drive"
os.environ["RCLONE_CONFIG_GDRIVE_SCOPE"] = "drive"
os.environ["RCLONE_CONFIG_GDRIVE_TOKEN"] = UserSecretsClient().get_secret("RCLONE_TOKEN").strip()
r = subprocess.run(["rclone", "listremotes"], capture_output=True, text=True)
if "gdrive:" not in r.stdout:
    raise RuntimeError("gdrive remote not found - check RCLONE_TOKEN secret.")
print(f"rclone remotes: {r.stdout.strip()}")

# Sync checkpoints from Drive (empty on first run is fine)
ckpt_dst = Path(KAGGLE_ROOT) / "checkpoints"
ckpt_dst.mkdir(parents=True, exist_ok=True)
result = subprocess.run(["rclone", "sync",
                         "gdrive:Colab Notebooks/TL-Bot/checkpoints/",
                         str(ckpt_dst), "--progress"])
if result.returncode != 0:
    print("Warning: checkpoint sync returned non-zero - continuing (may be first run).")
else:
    print("Checkpoint sync complete.")

# Pull dataset from Drive (skip if already extracted this session). Uses
# DATASET_NAME from cell 1 -- this rclone-based sync is Kaggle-specific and
# separate from remote_train.py's own sync_dataset(), so it must
# independently stay in sync with the configured name.
ds_dir = Path(KAGGLE_ROOT) / DATASET_NAME
if not ds_dir.exists():
    ds_zip = Path(KAGGLE_ROOT) / f"{DATASET_NAME}.zip"
    print(f"Pulling {DATASET_NAME}.zip from Drive ...")
    subprocess.run(["rclone", "copy",
                    f"gdrive:Colab Notebooks/TL-Bot/{DATASET_NAME}.zip",
                    str(Path(KAGGLE_ROOT)), "--progress"], check=True)
    with _zipfile.ZipFile(ds_zip, "r") as zf:
        zf.extractall(Path(KAGGLE_ROOT))
    ds_zip.unlink()
    print(f"Dataset ready: {ds_dir}")
else:
    print(f"Dataset already present: {ds_dir}")

---
## Cell 3 - Train
Clones or pulls the repo, symlinks the synced dataset in, then starts or resumes training.

In [ ]:
import os, subprocess, sys, json
from pathlib import Path as _Path

REPO_URL = "https://github.com/alexjade96/Discord-TL_Bot.git"
REPO_DIR = "/kaggle/working/Discord-TL_Bot"

# Checkpoint dir, mirroring train.py's scoping. Resolved once here so cell 4 can
# reuse it instead of repeating the rule.
_ALL = {"latin", "kana", "hangul", "cjk"}
_sel = _ALL if "all" in SCRIPTS else set(SCRIPTS)
if _sel >= _ALL:
    CKPT_DIR = _Path(KAGGLE_ROOT) / "checkpoints"
elif len(SCRIPTS) == 1:
    CKPT_DIR = _Path(KAGGLE_ROOT) / "checkpoints" / SCRIPTS[0]
else:
    CKPT_DIR = _Path(KAGGLE_ROOT) / "checkpoints" / "_".join(sorted(_sel))
# Mirrors remote_train.py's _make_ckpt_dir(): a non-default DATASET_NAME gets
# its own checkpoint dir so it can never land in and overwrite a
# default-dataset run's checkpoint.
if DATASET_NAME != "char-dataset":
    _suffix = DATASET_NAME[len("char-dataset"):].lstrip("-_") or DATASET_NAME
    CKPT_DIR = CKPT_DIR.parent / f"{CKPT_DIR.name}_{_suffix}"

os.makedirs(REPO_DIR, exist_ok=True)
if os.path.isdir(f"{REPO_DIR}/.git"):
    subprocess.run(["git", "-C", REPO_DIR, "pull", "--ff-only"], check=True)
else:
    subprocess.run(["git", "clone", REPO_URL, REPO_DIR], check=True)

# Symlink dataset into repo tree (Models/Datasets/ is gitignored, won't exist after clone).
_ds_src = str(_Path(KAGGLE_ROOT) / DATASET_NAME)
_ds_dst = f"{REPO_DIR}/Models/Datasets/{DATASET_NAME}"
if not os.path.exists(_ds_dst):
    os.makedirs(f"{REPO_DIR}/Models/Datasets", exist_ok=True)
    os.symlink(_ds_src, _ds_dst)
    print(f"Dataset linked: {_ds_src} -> {_ds_dst}")

# Print last training progress from progress.json before launching -- DO NOT REMOVE
# Wrapped: a display problem must never stop the training launch.
_prog = CKPT_DIR / "progress.json"
try:
    if _prog.exists():
        print("[progress]")
        for k, v in json.loads(_prog.read_text()).items():
            if not isinstance(v, (list, dict)):
                print(f"  {k}: {v}")
    else:
        print("[progress] No prior run found - starting fresh.")
except Exception as e:
    print(f"[progress] Could not read progress.json: {e}")


# Run a child process with its output streamed into the notebook.
#
# subprocess.run() without a pipe is useless here: IPython replaces sys.stdout at
# the Python level only, so a child inherits the kernel's real fd 1 and writes to
# the server log, not this cell. Pipe it and re-print through sys.stdout instead.
def _run_streamed(cmd):
    print("$ " + " ".join(str(c) for c in cmd) + "\n", flush=True)
    proc = subprocess.Popen(cmd, stdout=subprocess.PIPE, stderr=subprocess.STDOUT,
                            text=True, bufsize=1, errors="replace")
    tail = []
    for line in proc.stdout:
        sys.stdout.write(line)
        sys.stdout.flush()
        tail.append(line)
        del tail[:-40]
    rc = proc.wait()
    if rc != 0:
        raise RuntimeError(
            f"remote_train.py exited {rc}. Last {len(tail)} lines:\n" + "".join(tail))
    return rc


_cmd = [
    "python", "-u", f"{REPO_DIR}/Models/remote_train.py",
    "--skip-clone",
    "--scripts", *SCRIPTS,
    "--epochs", str(EPOCHS),
    "--scheduler", SCHEDULER,
    "--lr", str(LR),
    "--storage-root", KAGGLE_ROOT,
    "--skip-dataset",
    "--repo-dir", REPO_DIR,
    "--dataset-name", DATASET_NAME,
    "--sync-to", "gdrive:Colab Notebooks/TL-Bot/checkpoints/",
]
if RESUME:
    _cmd.append("--resume")
_run_streamed(_cmd)

---
## Cell 4 - Final Results
Run once cell 3 finishes. Prints the run summary and the last epoch's metrics from `progress.json`, plus `curves.png`.

Says so and stops if the run has not reached its last epoch. Fields are read from the JSON as they come, so metrics added to `train.py` show up without editing this cell.

The test-set report (per-class precision/recall, top-1/3/5, confused pairs) is printed at the end of cell 3 and is not saved to disk.

In [ ]:
import json

# Final results, read from progress.json in CKPT_DIR (resolved in cell 3).
# Keys come from the file, so metrics added to train.py appear without edits here.

_d = json.loads((CKPT_DIR / "progress.json").read_text())
_hist = _d.get("history", [])

if _d.get("completed", 0) < _d.get("total_epochs", 0):
    print(f"[results] Training unfinished: epoch {_d.get('completed')} of "
          f"{_d.get('total_epochs')}. Re-run once cell 3 completes.")
else:
    print("=" * 72)
    print(f" FINAL RESULTS   {CKPT_DIR}")
    print("=" * 72)
    for k, v in _d.items():
        if not isinstance(v, (list, dict)):
            print(f"  {k:<20}: {v}")

    if _hist:
        print(f"\n  --- last epoch ({_hist[-1].get('epoch')}) ---")
        for k, v in _hist[-1].items():
            print(f"  {k:<20}: {v}")

    if (CKPT_DIR / "curves.png").exists():
        from IPython.display import Image, display
        display(Image(filename=str(CKPT_DIR / "curves.png")))